In [2]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class AugmentedCTDataset(Dataset):
    def __init__(self, img_dir, target_size=256):
        self.paths = [os.path.join(img_dir, f) 
                      for f in os.listdir(img_dir) if f.endswith(".png")]
        self.target_size = target_size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (self.target_size, self.target_size))

        # Normalize
        img = img.astype("float32") / 255.0

        # Convert to PyTorch tensor: (C,H,W)
        img = torch.tensor(img).unsqueeze(0)

        # Dummy label (0 for now)
        label = torch.tensor(0).long()

        return img, label


In [3]:
train_dir = "C:/Users/sanja/Desktop/MRI_Project/Pleural Effusion/AUG_Dataset"

dataset = AugmentedCTDataset(train_dir)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

print("Total Training Samples:", len(dataset))


Total Training Samples: 420


In [4]:
import torch.nn as nn

class CTClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CTClassifier(num_classes=2).to(device)

loss_fn = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=1e-4)


In [6]:
for epoch in range(5):
    running_loss = 0
    correct = 0
    total = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        preds = model(imgs)
        loss = loss_fn(preds, labels)

        optim.zero_grad()
        loss.backward()
        optim.step()

        running_loss += loss.item()

        # accuracy
        _, predicted = torch.max(preds, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total

    print(f"Epoch {epoch}  Loss={epoch_loss:.4f}  Accuracy={epoch_acc:.2f}%")


Epoch 0  Loss=0.5787  Accuracy=100.00%
Epoch 1  Loss=0.4702  Accuracy=100.00%
Epoch 2  Loss=0.2933  Accuracy=100.00%
Epoch 3  Loss=0.1191  Accuracy=100.00%
Epoch 4  Loss=0.0390  Accuracy=100.00%


In [7]:
imgs, _ = next(iter(loader))
imgs_gpu = imgs.to(device)
preds = model(imgs_gpu)

print("Predicted classes:", preds.argmax(1).cpu().numpy())


Predicted classes: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
